
# Shapiro-Wilk Test for Normality

The **Shapiro-Wilk test** is used to test whether a sample is consistent with a normal distribution.

### Hypotheses

- **H₀:** The data come from a normal distribution.
- **H₁:** The data do not come from a normal distribution.

### Decision rule at α = 0.05

- **p ≤ 0.05** → Reject H₀ → evidence of non-normality.
- **p > 0.05** → Fail to reject H₀ → insufficient evidence of non-normality.

This notebook includes the Python implementation, visualization, decision making, multiple-column testing, and practical notes.


In [ ]:

# Install packages if needed:
# %pip install numpy pandas scipy matplotlib seaborn


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import shapiro

ALPHA = 0.05
np.random.seed(42)


## 1. Create Example Data

In [ ]:

# Example: normally distributed observations
data = np.random.normal(loc=100, scale=15, size=100)
df = pd.DataFrame({"Value": data})

df.head()


In [ ]:

print("Number of observations:", len(df))
display(df["Value"].describe().round(4))


## 2. Visualize the Data

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df["Value"], kde=True, ax=axes[0])
axes[0].set_title("Histogram with KDE")
axes[0].set_xlabel("Value")

stats.probplot(df["Value"], dist="norm", plot=axes[1])
axes[1].set_title("Normal Q-Q Plot")

plt.tight_layout()
plt.show()


## 3. Perform the Shapiro-Wilk Test

In [ ]:

W_statistic, p_value = shapiro(df["Value"])

print(f"Shapiro-Wilk W statistic = {W_statistic:.6f}")
print(f"p-value                  = {p_value:.6f}")
print(f"Significance level α     = {ALPHA}")


In [ ]:

if p_value <= ALPHA:
    decision = "Reject H0"
    interpretation = (
        "There is statistically significant evidence that the data "
        "are not normally distributed."
    )
else:
    decision = "Fail to reject H0"
    interpretation = (
        "There is insufficient evidence to conclude that the data "
        "are not normally distributed."
    )

print("Decision:", decision)
print("Interpretation:", interpretation)


## 4. Reusable Shapiro-Wilk Function

In [ ]:

def shapiro_normality_test(data, alpha=0.05):
    """Run Shapiro-Wilk test and return the statistical decision."""
    data = pd.Series(data).dropna()

    if len(data) < 3:
        raise ValueError("Shapiro-Wilk test requires at least 3 observations.")

    W, p = shapiro(data)

    if p <= alpha:
        decision = "Reject H0"
        conclusion = "Evidence suggests the data are not normally distributed."
    else:
        decision = "Fail to reject H0"
        conclusion = (
            "Insufficient evidence to conclude that the data are "
            "not normally distributed."
        )

    return {
        "n": len(data),
        "W_statistic": W,
        "p_value": p,
        "alpha": alpha,
        "decision": decision,
        "conclusion": conclusion
    }


In [ ]:

result = shapiro_normality_test(df["Value"], alpha=ALPHA)

for key, value in result.items():
    print(f"{key}: {value}")


## 5. Compare Normal and Non-Normal Data

In [ ]:

normal_data = np.random.normal(100, 15, 100)
non_normal_data = np.random.exponential(scale=20, size=100)

normal_result = shapiro_normality_test(normal_data, ALPHA)
non_normal_result = shapiro_normality_test(non_normal_data, ALPHA)

comparison = pd.DataFrame(
    [normal_result, non_normal_result],
    index=["Normal Data", "Non-Normal Data"]
)

display(comparison)


## 6. Run Shapiro-Wilk on Multiple Variables

In [ ]:

multi_df = pd.DataFrame({
    "Normal_Variable": np.random.normal(50, 10, 100),
    "Skewed_Variable": np.random.exponential(10, 100),
    "Another_Normal": np.random.normal(0, 1, 100)
})

results = []

for column in multi_df.columns:
    result = shapiro_normality_test(multi_df[column], ALPHA)
    result["Variable"] = column
    results.append(result)

results_df = pd.DataFrame(results)[[
    "Variable",
    "n",
    "W_statistic",
    "p_value",
    "alpha",
    "decision",
    "conclusion"
]]

display(results_df)



## 7. Interpretation and Practical Notes

### Decision table

| p-value | Decision | Interpretation |
|---|---|---|
| p ≤ 0.05 | Reject H₀ | Evidence of non-normality |
| p > 0.05 | Fail to reject H₀ | Insufficient evidence of non-normality |

**Important:** Failing to reject H₀ does not prove that the data are perfectly normal.

For ANOVA or regression, normality of the **residuals** is usually more relevant than normality of every raw variable.

Example:

```python
residuals = model.resid
shapiro_normality_test(residuals)
```

Also combine the Shapiro-Wilk result with a histogram, Q-Q plot, sample size, and domain knowledge. Very large samples can make the test sensitive to small deviations from normality.



# Final Workflow

```text
Data
  ↓
Handle missing values
  ↓
Histogram + Q-Q Plot
  ↓
Shapiro-Wilk Test
  ↓
p-value
  ├── p ≤ 0.05 → Reject H₀ → Evidence of non-normality
  │
  └── p > 0.05 → Fail to reject H₀
                    → Insufficient evidence of non-normality
```

### Key takeaway

> **Shapiro-Wilk tests whether the sample provides evidence against normality.**
